In [69]:
import glob
import os
import re
import pandas as pd

1. Перетворення файлy на табличний вигляд

In [70]:
def process_nbu_file(file_path, report_date):
    # 1. Зчитуємо Excel без заголовків
    df = pd.read_excel(file_path, sheet_name='tabl', header=None)

    # 2. Витягуємо рівні шапки (рядки 5, 6, 7 у файлі Excel -> індекси 4, 5, 6 у Python)
    borrower_types = df.iloc[4].ffill()  # Фізичні особи / Юридичні особи
    currencies = df.iloc[5].ffill()      # усього / національна валюта / іноземна валюта
    risk_classes = df.iloc[6]            # 1, 2, 3, 4, 5

    # 3. Вибираємо лише рядки з даними (починаючи з індексу 8 у Python, тобто з рядка 9 Excel)
    data_df = df.iloc[8:].copy()

    # Стовпчик B (індекс 1) - Банк, Стовпчик C (індекс 2) - Показник
    data_df[1] = data_df[1].ffill()  # Заповнюємо назву банку для другого рядка (Кредитний ризик)

    # ПРИВЕДЕННЯ ДО СТРІЧКИ: Додаємо .astype(str) для уникнення AttributeError
    data_df[1] = data_df[1].astype(str).str.replace(r'^\d+\s+', '', regex=True)
    # 4. Формуємо унікальні назви колонок для значень
    column_names = []
    for i in range(len(df.columns)):
        if i == 0:
            column_names.append('N_zp')
        elif i == 1:
            column_names.append('Bank')
        elif i == 2:
            column_names.append('Metric')
        else:
            b_type = str(borrower_types[i]).strip()
            curr = str(currencies[i]).strip()
            r_class = str(risk_classes[i]).strip()
            column_names.append(f"{b_type}__{curr}__Клас_{r_class}")

    data_df.columns = column_names

    # Видаляємо непотрібний стовпчик N_zp та порожні рядки
    data_df = data_df.drop(columns=['N_zp']).dropna(subset=['Bank', 'Metric'])

    # 5. Перетворюємо з широкого формату в довгий (Unpivot / Melt)
    melted_df = data_df.melt(
        id_vars=['Bank', 'Metric'],
        var_name='Combined_Category',
        value_name='Value'
    )

    # Розбиваємо створену колонку назад на окремі виміри
    split_cols = melted_df['Combined_Category'].str.split('__', expand=True)
    melted_df['Borrower_Type'] = split_cols[0]
    melted_df['Currency'] = split_cols[1]
    melted_df['Risk_Class'] = split_cols[2]

    # Прибираємо тимчасову колонку та додаємо дату
    melted_df = melted_df.drop(columns=['Combined_Category'])
    melted_df['Report_Date'] = pd.to_datetime(report_date)

    # Очищаємо числові значення від можливих пробілів чи тексу
    melted_df['Value'] = pd.to_numeric(melted_df['Value'], errors='coerce').fillna(0)

    return melted_df


file_paths = sorted(glob.glob('data/Kred_class_*.xlsx'))
all_dfs = []


for file in file_paths:
    # Витягуємо дату з назви файлу
    date_str = re.search(r'\d{4}-\d{2}-\d{2}', file).group(0)

    # Обробляємо один місяць
    df_clean = process_nbu_file(file, date_str)
    all_dfs.append(df_clean)

# Об'єднуємо все в один чистий датасет!
df_final = pd.concat(all_dfs, ignore_index=True)

# Перевіримо результат
df_final.head(10)

,Bank,Metric,Value,Borrower_Type,Currency,Risk_Class,Report_Date
0,"2 АТ ""Укрексімбанк""",Сума кредитної заборгованості,2.304624e+05,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
1,"2 АТ ""Укрексімбанк""",Кредитний ризик,1.947169e+04,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
2,"6 АТ ""Ощадбанк""",Сума кредитної заборгованості,2.405892e+07,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
3,"6 АТ ""Ощадбанк""",Кредитний ризик,1.517360e+06,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
4,"29 АТ ""БАНК АЛЬЯНС""",Сума кредитної заборгованості,3.716481e+05,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
5,"29 АТ ""БАНК АЛЬЯНС""",Кредитний ризик,4.310781e+04,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
6,"36 АТ ""Райффайзен Банк""",Сума кредитної заборгованості,6.670036e+06,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
7,"36 АТ ""Райффайзен Банк""",Кредитний ризик,2.437754e+05,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
8,"43 АТ ""АЛЬТБАНК""",Сума кредитної заборгованості,2.193276e+04,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
9,"43 АТ ""АЛЬТБАНК""",Кредитний ризик,3.076504e+02,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01


In [71]:
# Зберігаємо у CSV з кодуванням utf-8-sig (щоб кирилиця правильно відкривалася в Excel)
df_final.to_csv('data_2025.csv', index=False, encoding='utf-8-sig')

In [72]:
# Подивитися всі унікальні дати та кількість рядків для кожної
print(df_final['Report_Date'].value_counts().sort_index())

Report_Date
2025-02-01    12810
2025-03-01    12810
2025-04-01    12600
2025-05-01    12600
2025-06-01    12600
2025-07-01    12600
2025-08-01    12600
2025-09-01    12600
2025-10-01    12600
2025-11-01    12600
2025-12-01    12600
2026-01-01    12390
Name: count, dtype: int64


2. Прибираємо дублювання

В даних не треба враховувати:
- національна валюта та іноземна (вони в сумі дають усього)
- у тому числі непрацюючі (прибераємо те що у файлі, ми самі нище це визначимо. якщо не відфільтрувати той самий борг додасться двічі)

3. Ділимо на працюючі та непрацюючі:

Група 1 (5 класів): Фізичні особи, Банки, Бюджетні установи, SPE

Працюючі: 1–4 класи

Непрацюючі (NPL): 5 клас

Група 2 (10 класів): Юридичні особи (крім банків, бюджету, SPE)

Працюючі: 1–9 класи

Непрацюючі (NPL): 10 клас



In [76]:
# 1. Створюємо копію фінального датасету
df_prep = df_final.copy()

# 2. ФІЛЬТРУЄМО ЗАЙВІ РЯДКИ (Залишаємо strictly 'усього' та відсікаємо меморандумні колонки)
df_filtered = df_prep[
    (df_prep['Currency'].str.lower().str.strip() == 'усього')
    & (
        ~df_prep['Risk_Class']
        .str.lower()
        .str.contains('непрацююч', na=False)
    )
].copy()


# 3. Застосовуємо точну функцію класифікації під ТЗ
def classify_npl_exact(row):
    r_class = str(row['Risk_Class']).lower().strip()
    b_type = str(row['Borrower_Type']).lower().strip()

    # 1. ЮРИДИЧНІ ОСОБИ (Шукаємо 'юрид' або 'корпор', але виключаємо випадок, коли це чисто БАНКИ або БЮДЖЕТ)
    # Зверни увагу: перевіряємо, чи НЕ починається назва з 'банк' або 'бюджет'
    if ('юрид' in b_type or 'корпор' in b_type) and not (
        b_type.startswith('банк') or b_type.startswith('бюджет')
    ):
        if '10' in r_class:
            return 'Непрацюючий (NPL)'
        return 'Працюючий'

    # 2. ФІЗИЧНІ ОСОБИ, БАНКИ, БЮДЖЕТНІ УСТАНОВИ, SPE -> NPL це СТРОГО 5 клас
    if any(
        k in b_type
        for k in [
            'фізич',
            'компанії спеціального призначення',
            ' spe',
            'банк',
            'бюджет',
        ]
    ):
        if '5' in r_class:
            return 'Непрацюючий (NPL)'
        return 'Працюючий'

    # 3. Резервна перевірка
    if '10' in r_class:
        return 'Непрацюючий (NPL)'

    return 'Працюючий'


df_filtered['Asset_Status'] = df_filtered.apply(classify_npl_exact, axis=1)

# 4. Виводимо перші 20 рядків для перевірки
df_filtered[
    ['Bank', 'Borrower_Type', 'Risk_Class', 'Asset_Status', 'Value']
].head(20)

,Bank,Borrower_Type,Risk_Class,Asset_Status,Value
0,"2 АТ ""Укрексімбанк""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,2.304624e+05
1,"2 АТ ""Укрексімбанк""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,1.947169e+04
2,"6 АТ ""Ощадбанк""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,2.405892e+07
3,"6 АТ ""Ощадбанк""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,1.517360e+06
4,"29 АТ ""БАНК АЛЬЯНС""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,3.716481e+05
5,"29 АТ ""БАНК АЛЬЯНС""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,4.310781e+04
6,"36 АТ ""Райффайзен Банк""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,6.670036e+06
7,"36 АТ ""Райффайзен Банк""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,2.437754e+05
8,"43 АТ ""АЛЬТБАНК""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,2.193276e+04
9,"43 АТ ""АЛЬТБАНК""","Фізичні особи, у тому числі ФОП",Клас_1,Працюючий,3.076504e+02


In [80]:
df_filtered.to_csv('data_pere.csv', index=False, encoding='utf-8-sig')

In [81]:
# Створюємо додатковий прапорець для Ощадбанку
df_filtered['Is_Oschad'] = df_filtered['Bank'].str.contains(
    'Ощадбанк', case=False, na=False
)

df_filtered.to_csv('data_ready_for_charts.csv', index=False)